### HA_control
Reads the weekly load sheet (`Weekly_load_stats.csv`), checks it, and tells a layer notebook which tables to run.

Load it with `%run ./HA_control` as the only line of its own cell. It defines functions only, so it is safe to `%run`.


In [0]:
# HA_control: shared helper, load it from each layer notebook with  %run ./HA_control
import logging
from pyspark.sql import functions as F
logger = logging.getLogger("control")

CONTROL_COLS = ["source", "layer", "table_type", "table_name", "load_type", "enable"]
VALID_LAYERS = {"staging", "ingestion", "masked"}
VALID_TYPES  = {"fact", "dim"}
VALID_LOADS  = {"full_load", "inc_load"}
# what each layer's notebook does today (table_type -> load behaviour)
CODE_LOAD = {
    "staging":   {"fact": "full_load", "dim": "full_load"},   # both overwritten every run
    "ingestion": {"fact": "inc_load",  "dim": "full_load"},   # fact = change-feed stream, dim = full read + MERGE
    "masked":    {"fact": "inc_load",  "dim": "full_load"},
}

def load_control(path):
    df = spark.read.option("header", True).csv(path)
    df = df.toDF(*[c.strip().lower() for c in df.columns])
    missing = [c for c in CONTROL_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"control sheet is missing columns: {missing}")
    for c in CONTROL_COLS:                       # trim + lower-case so 'Y ', 'y' and 'Full_Load' all compare cleanly
        df = df.withColumn(c, F.lower(F.trim(F.col(c))))
    return df.select(*CONTROL_COLS).collect()

def validate_control(rows):
    errors, seen = [], set()
    for r in rows:
        key = (r["source"], r["layer"], r["table_name"])
        if key in seen:
            errors.append(f"duplicate row: {key}")
        seen.add(key)
        if r["layer"] not in VALID_LAYERS:     errors.append(f"{key}: layer must be one of {sorted(VALID_LAYERS)}")
        if r["table_type"] not in VALID_TYPES: errors.append(f"{key}: table_type must be one of {sorted(VALID_TYPES)}")
        if r["load_type"] not in VALID_LOADS:  errors.append(f"{key}: load_type must be Full_Load or Inc_Load")
        if r["enable"] not in ("y", "n"):      errors.append(f"{key}: enable must be Y or N, got {r['enable']!r}")
    if errors:
        raise ValueError("control sheet problems:\n  " + "\n  ".join(errors))

def get_run_list(path, layer, source, known_tables):
    """Enabled tables for one layer and source, dimensions first.
    known_tables = table names that layer's mapping knows about (lower-case)."""
    rows = load_control(path)
    validate_control(rows)
    todo = [r for r in rows if r["layer"] == layer and r["source"] == source.lower() and r["enable"] == "y"]
    if not todo:
        raise ValueError(f"nothing enabled in the control sheet for layer={layer!r}, source={source!r}")
    unknown = [r["table_name"] for r in todo if r["table_name"] not in known_tables]
    if unknown:
        raise ValueError(f"{layer}: control sheet tables not found in the mapping: {unknown}. "
                         f"The mapping knows: {sorted(known_tables)}")
    for r in todo:                                # warn, do not fail: the sheet and the notebook disagree
        code = CODE_LOAD[layer][r["table_type"]]
        if r["load_type"] != code:
            logger.warning(f"{layer}.{r['table_name']}: sheet says {r['load_type']} but the notebook runs it as {code}")
    return sorted(todo, key=lambda r: 0 if r["table_type"] == "dim" else 1)


from datetime import datetime
LOG_TABLE = None          # set in each layer's run cell from config["control"]["log_table"]

def log_run(run_id, layer, table_name, status, started, error=None):
    """One row per table per run. A logging problem must never hide the real load error."""
    try:
        (spark.createDataFrame(
            [(str(run_id), layer, table_name, status, started, datetime.now(), (error or "")[:500])],
            "run_id string, layer string, table_name string, status string, started timestamp, ended timestamp, error string")
         .write.mode("append").saveAsTable(LOG_TABLE))
    except Exception as e:
        logger.warning(f"could not write run log for {layer}.{table_name}: {e}")
